# Robotics Instability Detection — Relational Kernel

## Problem

In a multi-robot fleet, one robot undergoes gradual efficiency degradation
modelling actuator or motor wear. The dispatcher continues routing tasks to it
(scheduler inertia), its queue grows, and load redistributes to neighbors.
The cascade builds toward fleet-wide overload.

The challenge: detect the onset of this cascade before any robot hits its
queue capacity limit.

## Detection Approach

Rather than monitoring individual queue depths, we track the relational
structure of the fleet queue depth vector Q(t) = [Q_0, ..., Q_N-1].

As a cascade builds, load distribution becomes increasingly uneven —
relational gradients across the fleet amplify. Three operators from the
ABCRE invariant relational kernel detect this amplification:

```
A(x)[i]      = x[i] - mean(x)                               # gradient extraction
R(x, rho)[i] = x[i] + rho * (x[(i+1)%n] - x[(i-1)%n])     # antisymmetric circulation
C(x)[i]      = x[i] / (1 + |x[i]|)                         # bounded coherence
```

**Relational spread** (replaces np.var):

```
relational_spread(Q) = ||A(Q)||^2 / n
```

A extracts the gradient of each queue depth from the fleet mean.
The squared magnitude equals population variance — derived from first principles.

**Relational momentum** (replaces Kendall tau):

```
signal(W) = mean(|C(R(A(W), rho))|)
```

R amplifies directional trend via antisymmetric circulation.
C bounds the output to (-1, 1). The mean absolute value grows with
trend strength before any queue hits the failure threshold.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib.pyplot as plt

from rb_rel.network     import build_fleet, MAX_QUEUE_DEPTH, N
from rb_rel.disruption  import gradual_degradation
from rb_rel.simulation  import simulate
from rb_rel.instability import relational_spread, rolling_relational_spread, relational_momentum
from rb_rel.metrics     import compute_trigger_times

%matplotlib inline

T                = 300
T_DISRUPTION     = 100
DISRUPTION_BOT   = 2
SIGNAL_THRESHOLD = 0.10
MOMENTUM_WINDOW  = 30
RHO              = 0.4

## Simulation

- 6 robots, each with an empty task queue at t=0
- Global task rate of 60 units/step, dispatched proportionally to processing capacity
- processing_rate = 10.0/robot — exact baseline balance (delta Q = 0 pre-disruption)
- Local ring diffusion (alpha=0.10) redistributes queue load between neighbors
- At t=100, R2 begins gradual degradation at rate 0.005/step, floor e_min=0.2

In [ ]:
params, neighbors = build_fleet()
disruption        = gradual_degradation(DISRUPTION_BOT, T_DISRUPTION)
q_hist, e_hist    = simulate(params, neighbors, T=T, disruption_fn=disruption)

max_q    = q_hist.max(axis=1)
spread_s = np.array([relational_spread(q_hist[t]) for t in range(T)])
momentum = relational_momentum(spread_s, window=MOMENTUM_WINDOW, rho=RHO)

## Detection Result

In [ ]:
it, ft, lt = compute_trigger_times(
    max_q, momentum,
    signal_threshold=SIGNAL_THRESHOLD,
    overload_threshold=MAX_QUEUE_DEPTH,
)

sig_val = momentum[it] if it is not None else float('nan')
print(f"Relational momentum at detection:  {sig_val:.4f}  (threshold: {SIGNAL_THRESHOLD})")
print(f"Instability detected:              t = {it}")
print(f"Failure threshold crossed:         t = {ft}")
print(f"Lead time:                         {lt} steps")

## Visualization

- **Top:** Queue depth per robot over time. Red dashed = degradation onset.
  Green dotted = relational detection. Orange dotted = first failure.
  The gap between green and orange is the lead time the detector provides.

- **Bottom:** Relational momentum signal. Rises as fleet queue structure
  becomes increasingly uneven — before any single queue hits the failure threshold.

In [ ]:
t_axis = np.arange(T)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True)

for i in range(N):
    ax1.plot(t_axis, q_hist[:, i], lw=1.0, alpha=0.6, label=f'R{i}')
ax1.axvline(T_DISRUPTION, color='crimson', ls='--', alpha=0.7,
            label=f'Degradation onset (t={T_DISRUPTION})')
if it is not None:
    ax1.axvline(it, color='forestgreen', ls=':', lw=2,
                label=f'Instability detected (t={it})')
if ft is not None:
    ax1.axvline(ft, color='darkorange', ls=':', lw=2,
                label=f'Failure (t={ft})')
ax1.axhline(MAX_QUEUE_DEPTH, color='darkorange', ls='--', alpha=0.3)
ax1.set_ylabel('Queue Depth')
ax1.set_title('Robotics Instability Detection — Relational Kernel (ABCRE)')
ax1.legend(fontsize=7, ncol=2)

ax2.plot(t_axis, momentum, color='darkorange', lw=1.5,
         label=f'Relational momentum — mean(|C(R(A(W)))|) w={MOMENTUM_WINDOW}')
ax2.axhline(SIGNAL_THRESHOLD, color='forestgreen', ls='--', alpha=0.5,
            label=f'Detection threshold ({SIGNAL_THRESHOLD})')
if it is not None:
    ax2.axvline(it, color='forestgreen', ls=':', lw=2)
ax2.set_ylabel('Relational Momentum')
ax2.set_xlabel('Time Step')
ax2.legend(fontsize=8)

plt.tight_layout()
plt.show()

## Sensitivity Analysis

Lead time versus degradation rate, sweeping from slow wear to rapid failure.
Demonstrates that relational momentum detects cascade onset across a range
of degradation severities without parameter retuning.

In [ ]:
from rb_rel.disruption import E_MIN

deg_rates  = np.arange(0.002, 0.012, 0.001)
lead_times = []

for dr in deg_rates:
    def make_disruption(rate):
        def fn(efficiency, t):
            e = efficiency.copy()
            if t >= T_DISRUPTION:
                e[DISRUPTION_BOT] = max(E_MIN, 1.0 - rate * (t - T_DISRUPTION))
            return e
        return fn

    p, nbrs   = build_fleet()
    hist, _   = simulate(p, nbrs, T=T, disruption_fn=make_disruption(dr))
    mq        = hist.max(axis=1)
    sp        = np.array([relational_spread(hist[t]) for t in range(T)])
    mom       = relational_momentum(sp, window=MOMENTUM_WINDOW, rho=RHO)
    _, _, lt_ = compute_trigger_times(mq, mom,
                                      signal_threshold=SIGNAL_THRESHOLD,
                                      overload_threshold=MAX_QUEUE_DEPTH)
    lead_times.append(lt_ if lt_ is not None else 0)

plt.figure(figsize=(9, 4))
plt.plot(deg_rates, lead_times, 'o-', color='steelblue', lw=1.5)
plt.xlabel('Degradation Rate')
plt.ylabel('Lead Time (steps)')
plt.title('Relational Detection — Lead Time vs Degradation Rate')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()